<a href="https://colab.research.google.com/github/gnoejh/ict1022/blob/main/Transformer/4_sequence_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequence Models

### 1. Traditional Sequence Models: n-grams and Markov Models


**Objective**: Capture word sequences by analyzing fixed-length contexts, useful for generating text based on the likelihood of word sequences.

**Explanation**: 
- **n-grams**: A contiguous sequence of `n` items from text. For example, in a trigram model (n=3), each sequence of three words has an assigned probability based on frequency. However, n-grams struggle with long-term dependencies.
- **Markov Models**: Markov chains extend n-grams by using state transitions with probabilities, which is effective in text generation. It predicts the next word based on the current state (e.g., previous word).
    

In [6]:

from collections import defaultdict, Counter
import random

# Sample text for Markov chain
text = "The sun sets over the distant hills as a gentle breeze rustles through the leaves, carrying with it the fragrance of blooming flowers and the promise of a peaceful evening."
# Generate bigrams and compute probabilities
def generate_bigrams(text):
    words = text.split()
    bigrams = [(words[i], words[i+1]) for i in range(len(words)-1)]
    return bigrams

def markov_chain(bigrams):
    chain = defaultdict(Counter)
    for word1, word2 in bigrams:
        chain[word1][word2] += 1
    # Normalize counts to probabilities
    for word1 in chain:
        total_count = float(sum(chain[word1].values()))
        for word2 in chain[word1]:
            chain[word1][word2] /= total_count
    return chain

bigrams = generate_bigrams(text)
print("Bigrams:", bigrams)
chain = markov_chain(bigrams)
print("Markov Chain:", dict(chain))

# Generate text based on the Markov chain
def generate_text(chain, start_word, length=10):
    word = start_word
    text = [word]
    for _ in range(length - 1):
        if word not in chain:
            break
        next_words = list(chain[word].keys())
        probabilities = list(chain[word].values())
        word = random.choices(next_words, probabilities)[0]
        text.append(word)
    return ' '.join(text)

print("Generated Text:", generate_text(chain, start_word="hello"))
    

Bigrams: [('The', 'sun'), ('sun', 'sets'), ('sets', 'over'), ('over', 'the'), ('the', 'distant'), ('distant', 'hills'), ('hills', 'as'), ('as', 'a'), ('a', 'gentle'), ('gentle', 'breeze'), ('breeze', 'rustles'), ('rustles', 'through'), ('through', 'the'), ('the', 'leaves,'), ('leaves,', 'carrying'), ('carrying', 'with'), ('with', 'it'), ('it', 'the'), ('the', 'fragrance'), ('fragrance', 'of'), ('of', 'blooming'), ('blooming', 'flowers'), ('flowers', 'and'), ('and', 'the'), ('the', 'promise'), ('promise', 'of'), ('of', 'a'), ('a', 'peaceful'), ('peaceful', 'evening.')]
Markov Chain: {'The': Counter({'sun': 1.0}), 'sun': Counter({'sets': 1.0}), 'sets': Counter({'over': 1.0}), 'over': Counter({'the': 1.0}), 'the': Counter({'distant': 0.25, 'leaves,': 0.25, 'fragrance': 0.25, 'promise': 0.25}), 'distant': Counter({'hills': 1.0}), 'hills': Counter({'as': 1.0}), 'as': Counter({'a': 1.0}), 'a': Counter({'gentle': 0.5, 'peaceful': 0.5}), 'gentle': Counter({'breeze': 1.0}), 'breeze': Counter({'

### 2. Recurrent Neural Networks (RNNs)

- RNN, LSTN, GRU: https://www.analyticsvidhya.com/blog/2022/01/tutorial-on-rnn-lstm-gru-with-implementation/


**Objective**: Model sequences by retaining information from previous steps, enabling context across time steps.

**Explanation**: 
- RNNs process sequences element by element, retaining a hidden state that captures information from prior steps. They struggle with long sequences due to issues like the vanishing gradient problem.
    

In [7]:

import torch
import torch.nn as nn

# Sample RNN
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, hidden = self.rnn(x)
        out = self.fc(out[:, -1, :])  # Using the last output for prediction
        return out

# Example usage
input_size = 10  # Input dimension
hidden_size = 20  # Hidden layer size
output_size = 1  # Output dimension

# Dummy data
rnn = SimpleRNN(input_size, hidden_size, output_size)
inputs = torch.randn(5, 3, input_size)  # Batch size = 5, Sequence length = 3
outputs = rnn(inputs)
print("RNN Outputs:", outputs)


RNN Outputs: tensor([[ 0.1916],
        [ 0.1063],
        [-0.3980],
        [ 0.0356],
        [ 0.1532]], grad_fn=<AddmmBackward0>)


### 3. Long Short-Term Memory (LSTM) and Gated Recurrent Unit (GRU)


**Objective**: Overcome RNN limitations with gates to selectively retain or forget information.

**Explanation**:
- **LSTM**: Adds input, forget, and output gates to control the flow of information, addressing vanishing gradient issues.
- **GRU**: A simplified LSTM with fewer gates, offering similar performance with less computational cost.
    

In [8]:

# Sample LSTM
class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, (hidden, cell) = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

# Example usage
lstm = SimpleLSTM(input_size, hidden_size, output_size)
outputs = lstm(inputs)
print("LSTM Outputs:", outputs)
    

LSTM Outputs: tensor([[-0.0333],
        [ 0.0184],
        [ 0.0195],
        [-0.0539],
        [-0.0385]], grad_fn=<AddmmBackward0>)


In [9]:

# Sample GRU
class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleGRU, self).__init__()
        self.hidden_size = hidden_size
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, hidden = self.gru(x)
        out = self.fc(out[:, -1, :])
        return out

# Example usage
gru = SimpleGRU(input_size, hidden_size, output_size)
outputs = gru(inputs)
print("GRU Outputs:", outputs)
    

GRU Outputs: tensor([[-0.0909],
        [ 0.0600],
        [-0.2093],
        [-0.0107],
        [ 0.1127]], grad_fn=<AddmmBackward0>)


### 4. Introduction to the Attention Mechanism


**Objective**: Enable models to focus on relevant parts of input sequences, essential for handling longer dependencies.

**Explanation**: 
- Attention allows a model to “attend” to specific input parts when generating each output. It’s especially useful for tasks requiring selective referencing of input tokens (e.g., translation, summarization).
- https://newsletter.theaiedge.io/p/the-transformer-architecture-v2
    

In [10]:

# Simple attention mechanism
def attention(query, key, value):
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(key.size(-1), dtype=torch.float32))
    attn_weights = torch.nn.functional.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, value)
    return output, attn_weights

# Example inputs
query = torch.randn(5, 3, 8)  # Batch size = 5, Sequence length = 3, Embedding size = 8
key = torch.randn(5, 3, 8)
value = torch.randn(5, 3, 8)

# Apply attention
output, weights = attention(query, key, value)
print("Attention Output:", output)
print("Attention Weights:", weights)
    

Attention Output: tensor([[[ 0.5493, -0.5043,  0.2931, -0.7099, -0.4739,  0.6647, -0.4197,
           0.2334],
         [ 0.5426, -0.5120,  0.1402, -0.9613, -0.3338,  0.5905, -0.3014,
           0.2008],
         [ 0.5382, -0.5284,  0.1075, -0.9978, -0.3034,  0.5686, -0.2819,
           0.1813]],

        [[-0.4462, -1.1030, -0.1611, -0.7733, -0.0529,  0.4743,  1.1760,
           0.2262],
         [-0.6423, -1.1718, -0.2259, -0.5479, -0.2377,  0.1492,  1.0690,
           0.2367],
         [-0.3045, -0.9583, -0.0060, -1.0684,  0.1645,  0.7895,  0.8988,
           0.1431]],

        [[ 0.5012,  0.9355,  0.0831,  0.1326, -0.2858, -0.2005, -1.0446,
           0.0506],
         [ 0.8995,  0.9982,  0.8436,  0.3707, -0.6034, -0.3766, -1.3645,
           0.4632],
         [ 0.7049,  1.0487,  0.1273,  0.1135, -0.2787, -0.2404, -0.9617,
          -0.0148]],

        [[ 0.1680,  0.1495, -0.7991,  0.5406, -1.2738,  0.0256, -0.2487,
          -0.2481],
         [ 0.2266,  0.0772, -1.0279,  0.4015, 